# Витрина эквайринга: Excel-отчёт бизнеса + обогащение из lake

База — Excel-отчёт за месяц (1 строка = 1 `agr_id`).
Из lake добавляются: `ssp_ocrm`, `cdi_id`, `ogrn` (по `inn`).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

In [ ]:
excel_sources = [
    {'report_month': '2026-01-01', 'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

output_csv_path = './datamart_month_excel_enriched_q1.csv'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 1) Загрузка Excel

In [ ]:
def normalize_inn(value):
    if pd.isna(value):
        return None
    s = re.sub(r'\D+', '', str(value))
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for a in aliases:
        k = normalize_colname(a)
        if k in norm_map:
            return norm_map[k]
    return None


frames = []
for src in excel_sources:
    raw = pd.read_excel(src['path'], header=src['header'])

    col_inn    = pick_column(raw.columns, ['ИНН', 'INN'])
    col_agr_id = pick_column(raw.columns, ['ID договора', 'agr_id'])
    if col_inn is None or col_agr_id is None:
        raise ValueError(f'Не найдены ИНН/agr_id в {src["path"]}. Колонки: {list(raw.columns)}')

    raw['inn']          = raw[col_inn].apply(normalize_inn)
    raw['agr_id']       = raw[col_agr_id].astype(str).str.strip()
    raw['report_month'] = pd.to_datetime(src['report_month'])
    frames.append(raw)

excel_df = pd.concat(frames, ignore_index=True)

print(f'Excel (3 месяца): {len(excel_df):,} строк')
print(f'  по месяцам:')
print(excel_df.groupby('report_month').size())
print(f'Уникальных agr_id: {excel_df["agr_id"].nunique():,}')
print(f'Уникальных inn:    {excel_df["inn"].dropna().nunique():,}')
excel_df.head(5)

## 2) `ssp_ocrm` и `cdi_id` по `inn` (через OCRM/CDI)

In [ ]:
inn_values = sorted(excel_df['inn'].dropna().astype(str).unique().tolist())
inn_in = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_ocrm = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc,
               cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_in})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ocrm_df = imp.fetch(sql_ocrm)

if ocrm_df is None:
    ocrm_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'cdi_id'])

if not ocrm_df.empty:
    ocrm_df['inn']    = ocrm_df['inn'].astype(str)
    ocrm_df['cdi_id'] = ocrm_df['cdi_id'].astype(str)
    allowed = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    ocrm_df['ssp_ocrm'] = ocrm_df['ssp_ocrm'].where(ocrm_df['ssp_ocrm'].isin(allowed), None)
    ocrm_df = ocrm_df.drop_duplicates(subset=['inn'], keep='first')

print(f'OCRM/CDI: {len(ocrm_df):,} строк')
ocrm_df.head(5)

## 3) `ogrn` по `cdi_id` (CDI -> CFT -> R2)

In [ ]:
cdi_values = sorted(ocrm_df['cdi_id'].dropna().astype(str).unique().tolist())
cdi_values = [x for x in cdi_values if x and x != 'nan']
cdi_in = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_ogrn = f"""
with cft as (
  select
    cast(e.party_id as string) as cdi_id,
    cast(e.cmo_ext_party_source_id as string) as cft_id
  from cdiul.ext_id_org e
  where cast(e.party_id as string) in ({cdi_in})
    and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
)
select
  c.cdi_id,
  cast(corp.c_register_gos_reg_num_rec as string) as ogrn
from cft c
left join ods.scd1_z_cl_corp corp
  on cast(corp.id as string) = c.cft_id
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ogrn_df = imp.fetch(sql_ogrn)

if ogrn_df is None:
    ogrn_df = pd.DataFrame(columns=['cdi_id', 'ogrn'])

if not ogrn_df.empty:
    ogrn_df['cdi_id'] = ogrn_df['cdi_id'].astype(str)
    ogrn_df['ogrn']   = ogrn_df['ogrn'].astype(str)
    ogrn_df = ogrn_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'OGRN: {len(ogrn_df):,} строк')
ogrn_df.head(5)

## 4) Финальная сборка и сохранение

In [ ]:
final_df = excel_df.merge(ocrm_df, on='inn', how='left')
final_df = final_df.merge(ogrn_df, on='cdi_id', how='left')

print(f'Итого: {len(final_df):,} строк, {len(final_df.columns)} колонок')
print(f'  ssp_ocrm заполнено: {final_df["ssp_ocrm"].notna().sum():,}')
print(f'  cdi_id   заполнено: {final_df["cdi_id"].notna().sum():,}')
print(f'  ogrn     заполнено: {final_df["ogrn"].notna().sum():,}')

output_path = Path(output_csv_path)
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'CSV: {output_path.resolve()}')

final_df.head(5)